In [1]:
from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.en", cache_dir="~/thesis/RAG/data")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

In [2]:
%pip install pandas


[notice] A new release of pip is available: 23.3.1 -> 24.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [99]:
import pandas as pd
import requests
from pprint import pprint
import json
import shutil
import os

In [5]:
wiki_df = pd.DataFrame(ds["train"][:2000])

In [7]:
display(wiki_df)
column_names = ["id", "title"]



,id,url,title,text
0,12,https://en.wikipedia.org/wiki/Anarchism,Anarchism,Anarchism is a political philosophy and moveme...
1,39,https://en.wikipedia.org/wiki/Albedo,Albedo,Albedo (; ) is the fraction of sunlight that i...
2,290,https://en.wikipedia.org/wiki/A,A,"A, or a, is the first letter and the first vow..."
3,303,https://en.wikipedia.org/wiki/Alabama,Alabama,Alabama () is a state in the Southeastern regi...
4,305,https://en.wikipedia.org/wiki/Achilles,Achilles,"In Greek mythology, Achilles ( ) or Achilleus ..."
...,...,...,...,...
1995,4563,https://en.wikipedia.org/wiki/Battle%20of%20Ju...,Battle of Jutland,"The Battle of Jutland (, the Battle of the Ska..."
1996,4565,https://en.wikipedia.org/wiki/Bambara%20language,Bambara language,"Bambara, also known as Bamana (N'Ko script: ) ..."
1997,4566,https://en.wikipedia.org/wiki/Baku,Baku,"Baku (, ; ) is the capital and largest city o..."
1998,4567,https://en.wikipedia.org/wiki/Balalaika,Balalaika,"The balalaika (, ) is a Russian stringed music..."


In [87]:
def concat_records(dataframe, new_records):
    column_names = ["id", "title"]
    new_dataframe = pd.DataFrame.from_records(data=new_records,columns=column_names)
    if dataframe is None:
        return new_dataframe
    return pd.concat([dataframe, new_dataframe], ignore_index=True)

## Get top level pages

In [140]:
astronomy_query = {
    "action": "query",
    "list": "categorymembers",
    "cmtitle": "Category:Astronomy",
    "prop": "categories",
    "cmlimit": 500,
    "cmtype": "page",
    "format": "json"
}
res = requests.get("https://en.wikipedia.org/w/api.php", params=astronomy_query)
res_json = res.json()
batch_id = 0
iteration = 0
batch_size = 10_000
titles_df = None

In [141]:
pprint(res_json)

{'batchcomplete': '',
 'query': {'categorymembers': [{'ns': 0, 'pageid': 50650, 'title': 'Astronomy'},
                               {'ns': 0,
                                'pageid': 34809573,
                                'title': 'Glossary of astronomy'},
                               {'ns': 0,
                                'pageid': 3400190,
                                'title': 'Outline of astronomy'},
                               {'ns': 100,
                                'pageid': 1484425,
                                'title': 'Portal:Astronomy'},
                               {'ns': 0,
                                'pageid': 49002199,
                                'title': 'Advanced Scientific Data Format'},
                               {'ns': 0,
                                'pageid': 289860,
                                'title': 'Alignments of random points'},
                               {'ns': 0,
                                'pageid': 522773

In [142]:
write_path = '/Users/mattan/thesis/RAG/astronomy_pages'

In [143]:
if os.path.exists(write_path):
    shutil.rmtree(write_path)
os.mkdir(write_path)


while res_json.get("continue", False):
    iteration += 1
    records = [(elem['pageid'], elem["title"]) for elem in res_json["query"]["categorymembers"]]
    print(f"{len(records)} records in response.", end=" ")
    titles_df = concat_records(titles_df, records)
    if iteration % (batch_size // 500) == 0:
        print(f"\niteration {iteration}. flushing {len(titles_df)} records to batch {batch_id}")
        titles_df.to_csv(f'{write_path}/datapage_id_title_batch_{batch_id}.csv', index=False)
        titles_df = None
        batch_id += 1
    astronomy_query.update(res_json["continue"])
    print("continue params: ", end="")
    pprint(res_json["continue"])
    res = requests.get("https://en.wikipedia.org/w/api.php", params=astronomy_query)
    res_json = res.json()

if titles_df is None:
    records = [(elem['pageid'], elem["title"]) for elem in res_json["query"]["categorymembers"]]
    titles_df = concat_records(titles_df, records)
titles_df.to_csv(f'~/thesis/RAG/astronomy_pages/datapage_id_title_batch_{batch_id}.csv', index=False)

## Get Subcategories

In [83]:

astronomy_query = {
    "action": "query",
    "list": "categorymembers",
    "cmtitle": "Category:Astronomy",
    "prop": "categories",
    "cmlimit": 500,
    "cmtype": "subcat",
    "format": "json"
}
res = requests.get("https://en.wikipedia.org/w/api.php", params=astronomy_query)
res_json = res.json()
iteration = 0
batch_size = 10_000
titles_df = None

In [117]:
pprint(res_json)


200

In [ ]:
write_path = '/Users/mattan/thesis/RAG/astronomy_pages'

In [ ]:
if os.path.exists(write_path):
    shutil.rmtree(write_path)
os.mkdir(write_path)


while res_json.get("continue", False):
    iteration += 1
    records = [(elem['pageid'], elem["title"]) for elem in res_json["query"]["categorymembers"]]
    print(f"{len(records)} records in response.", end=" ")
    titles_df = concat_records(titles_df, records)
    if iteration % (batch_size // 500) == 0:
        print(f"\niteration {iteration}. flushing {len(titles_df)} records to batch {batch_id}")
        titles_df.to_csv(f'{write_path}/datapage_id_title_batch_{batch_id}.csv', index=False)
        titles_df = None
        batch_id += 1
    astronomy_query.update(res_json["continue"])
    print("continue params: ", end="")
    pprint(res_json["continue"])
    res = requests.get("https://en.wikipedia.org/w/api.php", params=astronomy_query)
    res_json = res.json()

if titles_df is None:
    records = [(elem['pageid'], elem["title"]) for elem in res_json["query"]["categorymembers"]]
    titles_df = concat_records(titles_df, records)
titles_df.to_csv(f'~/thesis/RAG/astronomy_pages/datapage_id_title_batch_{batch_id}.csv', index=False)

In [46]:
with open( "/Users/mattan/thesis/RAG/data/example_response.json", "w") as f:
    json.dump(res.json(), f, indent=4)



In [49]:
res.json()['continue']['cmcontinue']

KeyError: 'cmcontinue'

In [63]:
train_ds = ds['train'].filter(lambda example: example['id'] == '65462618')
test_ds = ds['test'].filter(lambda example: example['id'] == '65462618')
validation_ds = ds['validation'].filter(lambda example: example['id'] == '65462618')



Filter:   0%|          | 0/6407814 [00:00<?, ? examples/s]

KeyError: 'test'

In [57]:
print(train_ds)

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 0
})


In [ ]:
# def get_category_members(category, limit=500):
#     url = "https://en.wikipedia.org/w/api.php"
#     params = {
#         "action": "query",
#         "list": "categorymembers",
#         "cmtitle": f"Category:{category}",
#         "cmlimit": limit,
#         "format": "json"
#     }
# 
#     titles = []
#     while True:
#         response = requests.get(url, params=params).json()
#         members = response.get('query', {}).get('categorymembers', [])
#         titles.extend(member['title'] for member in members)
# 
#         if 'continue' in response:
#             params['cmcontinue'] = response['continue']['cmcontinue']
#         else:
#             break
# 
#     return titles
# 
# # Example usage
# category = "Physics"
# titles = get_category_members(category)
# for title in titles:
#     print(title)
